# Phase 2 — Generate & Analyze Teacher Translations

**Project:** Multi-Hypothesis Distillation (MHD) — English → Swahili  
**Teacher:** `facebook/nllb-200-distilled-600M` (swap to 1.3B for higher quality)  
**Run environment:** Kaggle (GPU P100 / T4)

---
### Decoding methods covered
| Method | M | Notes |
|--------|---|-------|
| Beam search | 1 | Greedy-best; baseline |
| Beam search | 10 | Top-10 beam hypotheses |
| Top-p sampling | 10 | Nucleus sampling p=0.7 |
| Top-k sampling | 10 | Top-k=10 sampling |
| **DBS** | **10** | Diverse Beam Search (`num_beam_groups`) |
| **MBR** | **10** | Minimum Bayes Risk — sample 50, re-rank by peer BLEU |

---
### Pipeline
0. Kaggle zip extraction (auto-detect your dataset upload)  
1. Setup & imports  
2. Load source corpus  
3. Load NLLB teacher model  
4. Generate synthetic translations (all 6 jobs)  
5. Diversity analysis + self-BLEU  
6. Save CSV + plots  
7. *(Optional)* Evaluate teacher on FLORES devtest  
8. Final summary  

## 0. Kaggle Dataset / Zip Extraction

Upload your project zip as a Kaggle Dataset (or via the `+Add Data` button).  
Set `ZIP_INPUT_PATH` to its location. The cell will unzip it into `/kaggle/working/MHD2/` and set `PROJECT_ROOT` automatically.

If you are running locally (or the zip is already extracted), set `ZIP_INPUT_PATH = None`.

In [1]:
import zipfile, os, sys
from pathlib import Path

# ------------------------------------------------------------------
# SET THIS to your zip path on Kaggle, e.g.:
#   /kaggle/input/<dataset-name>/MHD2.zip
# Leave as None if you are running locally or already extracted.
# ------------------------------------------------------------------
ZIP_INPUT_PATH = None   # <-- CHANGE ME on Kaggle

# Where to extract (Kaggle writable space)
EXTRACT_ROOT = Path("/kaggle/working")

if ZIP_INPUT_PATH is not None:
    zip_path = Path(ZIP_INPUT_PATH)
    assert zip_path.exists(), f"Zip not found: {zip_path}"
    print(f"Extracting {zip_path.name} → {EXTRACT_ROOT} ...")
    with zipfile.ZipFile(zip_path, "r") as z:
        z.extractall(EXTRACT_ROOT)
    # Find the top-level project folder (first dir inside zip)
    with zipfile.ZipFile(zip_path, "r") as z:
        top_dirs = {Path(n).parts[0] for n in z.namelist() if "/" in n}
    if top_dirs:
        PROJECT_ROOT = EXTRACT_ROOT / sorted(top_dirs)[0]
    else:
        PROJECT_ROOT = EXTRACT_ROOT
    print(f"PROJECT_ROOT set to: {PROJECT_ROOT}")
else:
    # Running locally: notebook is inside notebooks/, project root is one level up
    PROJECT_ROOT = Path("..").resolve()
    print(f"Running locally. PROJECT_ROOT = {PROJECT_ROOT}")

# Add project root to sys.path for any local modules
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

Running locally. PROJECT_ROOT = C:\Users\nirmi\Desktop\MHD2


## 1. Setup & Imports

In [2]:
# Install / upgrade required libraries if not already present (safe on Kaggle)
import importlib, subprocess

def _pip(*pkgs):
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", *pkgs])

if importlib.util.find_spec("sacrebleu") is None:
    _pip("sacrebleu")
if importlib.util.find_spec("transformers") is None:
    _pip("transformers", "sentencepiece", "protobuf")
if importlib.util.find_spec("tqdm") is None:
    _pip("tqdm")

In [3]:
import json, random, csv, math, shutil, collections
from collections import Counter, defaultdict

import pandas as pd
import torch
from tqdm.auto import tqdm
import matplotlib
matplotlib.use("Agg")   # non-interactive — required on Kaggle
import matplotlib.pyplot as plt
import sacrebleu
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM

print("All imports OK")

c:\Users\nirmi\Desktop\MHD2\venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


All imports OK


### Paths

In [4]:
DATA_DIR      = PROJECT_ROOT / "data"
PROCESSED_DIR = DATA_DIR / "processed"
SYNTHETIC_DIR = DATA_DIR / "synthetic"
FLORES_DIR    = DATA_DIR / "flores"
RESULTS_DIR   = PROJECT_ROOT / "results"
PLOTS_DIR     = RESULTS_DIR / "plots"

for d in [SYNTHETIC_DIR, RESULTS_DIR, PLOTS_DIR]:
    d.mkdir(parents=True, exist_ok=True)

print("Paths OK")
print(f"  processed : {PROCESSED_DIR}")
print(f"  synthetic : {SYNTHETIC_DIR}")
print(f"  flores    : {FLORES_DIR}")
print(f"  results   : {RESULTS_DIR}")

Paths OK
  processed : C:\Users\nirmi\Desktop\MHD2\data\processed
  synthetic : C:\Users\nirmi\Desktop\MHD2\data\synthetic
  flores    : C:\Users\nirmi\Desktop\MHD2\data\flores
  results   : C:\Users\nirmi\Desktop\MHD2\results


### Configuration

In [5]:
SRC_LANG      = "eng_Latn"
TGT_LANG      = "swh_Latn"

# 600M: ~2.4 GB VRAM — good for Kaggle free GPU
# 1.3B: ~5.3 GB VRAM — switch here for higher quality
TEACHER_MODEL = "facebook/nllb-200-distilled-600M"

SAMPLE_SIZE   = 10_000   # source sentences to translate
MAX_LENGTH    = 128      # max tokens to generate
BATCH_SIZE    = 4        # reduce to 2 or 1 if CUDA OOM

# MBR-specific: pool size to sample before re-ranking
MBR_POOL_SIZE = 50

# Set True to regenerate files that already exist
OVERWRITE = False

# ---------------------------------------------------------------------------
# All generation jobs
# method, M, output filename
# ---------------------------------------------------------------------------
JOBS = [
    ("beam",  1,  "eng_swh_beam_M1.jsonl"),
    ("beam",  10, "eng_swh_beam_M10.jsonl"),
    ("top_p", 10, "eng_swh_top_p_M10.jsonl"),
    ("top_k", 10, "eng_swh_top_k_M10.jsonl"),
    ("dbs",   10, "eng_swh_dbs_M10.jsonl"),
    ("mbr",   10, "eng_swh_mbr_M10.jsonl"),
]

print("Config OK. Jobs:")
for m, M, f in JOBS:
    print(f"  {m:<8} M={M:<3}  → {f}")

Config OK. Jobs:
  beam     M=1    → eng_swh_beam_M1.jsonl
  beam     M=10   → eng_swh_beam_M10.jsonl
  top_p    M=10   → eng_swh_top_p_M10.jsonl
  top_k    M=10   → eng_swh_top_k_M10.jsonl
  dbs      M=10   → eng_swh_dbs_M10.jsonl
  mbr      M=10   → eng_swh_mbr_M10.jsonl


## 2. Load Source Corpus

In [6]:
corpus_path = PROCESSED_DIR / "eng_clean_10k.txt"
assert corpus_path.exists(), f"Source corpus not found: {corpus_path}"

with open(corpus_path, encoding="utf-8") as f:
    all_lines = [line.strip() for line in f if line.strip()]

sources = all_lines[:SAMPLE_SIZE]
print(f"Loaded {len(sources):,} source sentences")
print("\nFirst 5:")
for i, s in enumerate(sources[:5]):
    print(f"  [{i}] {s}")

Loaded 10,000 source sentences

First 5:
  [0] YES! I’ve actually gone without shampoo and soap for about 4 or 5 months now. My hair is great and I don’t have any offensive body-odors on a daily basis. If I plan to sweat a lot on a particular day, I just splash some tea tree oil in my ‘pits and I’m good to go!
  [1] should not have wandered there. It is no place for maids, nor for any
  [2] 3. Do you have a personal code of morals or ethics? If so, how did that begin? What would it take to compromise it? No morals or ethics whatsoever.
  [3] “I don’t know dude, I try not to think about it too much. But yeah, there’s always a down side to the short lived excitement of each babe you pay for. I’d hate to think how much I’ve blown on booze and whores man” said mark shaking his head.
  [4] evening, in the heat of discussion, "For Heaven's sake! the Berlin women


## 3. Load NLLB Teacher Model

In [7]:
# fp16 halves VRAM on CUDA. Never use fp16 on CPU (produces NaN).
device   = "cuda" if torch.cuda.is_available() else "cpu"
use_fp16 = (device == "cuda")

print(f"Device : {device}  |  fp16: {use_fp16}  |  Model: {TEACHER_MODEL}")

tokenizer = AutoTokenizer.from_pretrained(TEACHER_MODEL)
tokenizer.src_lang = SRC_LANG

model = AutoModelForSeq2SeqLM.from_pretrained(
    TEACHER_MODEL,
    torch_dtype=torch.float16 if use_fp16 else torch.float32,
)
model.to(device)
model.eval()

# Forces the decoder to produce Swahili output
forced_bos_token_id = tokenizer.convert_tokens_to_ids(TGT_LANG)
print(f"forced_bos_token_id ({TGT_LANG}): {forced_bos_token_id}")

Device : cuda  |  fp16: True  |  Model: facebook/nllb-200-distilled-600M


[transformers] `torch_dtype` is deprecated! Use `dtype` instead!
Loading weights: 100%|██████████| 512/512 [00:02<00:00, 187.05it/s]


forced_bos_token_id (swh_Latn): 256168


## 4. Generation Helpers

### 4a. Core generator (beam / top-p / top-k / DBS)

**Diverse Beam Search (DBS):** uses `num_beam_groups` in HuggingFace to partition the beam into groups that are penalised for being too similar to each other (controlled by `diversity_penalty`). This gives M hypotheses that are structurally more varied than standard beam search.

In [8]:
def _build_gen_kwargs(method, M, max_length):
    """
    Return model.generate() kwargs for a given decoding method.

    method options
    --------------
    beam   : standard beam search  (do_sample=False, num_beams=max(10,M))
    top_p  : nucleus sampling      (do_sample=True,  top_p=0.7)
    top_k  : top-k sampling        (do_sample=True,  top_k=10)
    dbs    : Diverse Beam Search   (num_beam_groups=M, diversity_penalty=0.8)

    CUDA OOM tips
    -------------
    - Reduce BATCH_SIZE to 2 or 1
    - Reduce MAX_LENGTH to 64
    - Use 600M model instead of 1.3B
    - For DBS with large M, reduce num_beam_groups
    """
    base = dict(max_length=max_length, forced_bos_token_id=forced_bos_token_id)

    if method == "beam":
        return {**base,
                "do_sample": False,
                "num_beams": max(10, M),
                "num_return_sequences": M}

    elif method == "top_p":
        return {**base,
                "do_sample": True,
                "top_p": 0.7,
                "num_beams": 1,
                "num_return_sequences": M}

    elif method == "top_k":
        return {**base,
                "do_sample": True,
                "top_k": 10,
                "num_beams": 1,
                "num_return_sequences": M}

    elif method == "dbs":
        # num_beams must be divisible by num_beam_groups
        # We use num_beams = M * 2 and num_beam_groups = M so each group
        # gets 2 beams, giving M diverse hypotheses.
        num_beams = M * 2
        return {**base,
                "do_sample": False,
                "num_beams": num_beams,
                "num_beam_groups": M,
                "diversity_penalty": 0.8,
                "num_return_sequences": M}

    else:
        raise ValueError(f"Unknown method: {method!r}. "
                         "Choose: beam | top_p | top_k | dbs")


def generate_translations(sources, method, M,
                          batch_size=BATCH_SIZE, max_length=MAX_LENGTH):
    """
    Generate M Swahili hypotheses per source sentence.
    Yields one dict per hypothesis:
      {src_id, src, hyp_id, tgt, method, M}

    Works for methods: beam, top_p, top_k, dbs.
    For mbr, use generate_mbr_translations() below.
    """
    gen_kwargs = _build_gen_kwargs(method, M, max_length)

    for batch_start in range(0, len(sources), batch_size):
        batch_srcs = sources[batch_start: batch_start + batch_size]

        inputs = tokenizer(
            batch_srcs,
            return_tensors="pt",
            padding=True,
            truncation=True,
            max_length=max_length,
        ).to(device)

        with torch.no_grad():
            outputs = model.generate(**inputs, **gen_kwargs)

        decoded = tokenizer.batch_decode(outputs, skip_special_tokens=True)
        # decoded shape: batch_size * M (interleaved)

        for i, src in enumerate(batch_srcs):
            src_id = batch_start + i
            for hyp_id in range(M):
                yield {
                    "src_id": src_id,
                    "src":    src,
                    "hyp_id": hyp_id,
                    "tgt":    decoded[i * M + hyp_id],
                    "method": method,
                    "M":      M,
                }

print("generate_translations() defined (beam / top_p / top_k / dbs).")

generate_translations() defined (beam / top_p / top_k / dbs).


### 4b. MBR (Minimum Bayes Risk) generator

**MBR decoding** selects the hypothesis that has the highest expected utility (here: BLEU) against all other hypotheses in a sampled pool.

**Algorithm for M=10 outputs:**
1. Sample a pool of `MBR_POOL_SIZE` (default 50) candidates using top-p sampling.
2. For each candidate `h_i`, compute its average sentence-BLEU against all other candidates in the pool.
3. Score = `mean_j≠i BLEU(h_i, h_j)`.
4. Return the top-M candidates ranked by this score.

This gives hypotheses that are *consensus* translations — neither the single best-beam output nor a random sample, but the translations most supported by the model's distribution.

In [9]:
def _mbr_score_pool(candidates):
    """
    Score each candidate by its average sentence-BLEU
    against all other candidates in the pool.

    Returns list of (score, candidate) sorted descending.
    """
    n = len(candidates)
    scores = []
    for i, cand in enumerate(candidates):
        refs = [candidates[j] for j in range(n) if j != i]
        # sacrebleu.sentence_bleu(hypothesis, list_of_references)
        bleu = sacrebleu.sentence_bleu(cand, refs).score
        scores.append((bleu, cand))
    scores.sort(key=lambda x: x[0], reverse=True)
    return scores


def generate_mbr_translations(sources, M,
                               pool_size=MBR_POOL_SIZE,
                               batch_size=BATCH_SIZE,
                               max_length=MAX_LENGTH):
    """
    MBR decoding: sample a pool, re-rank by peer BLEU, return top-M.

    pool_size >= M (recommend pool_size = 5*M for stable estimates).
    Yields one dict per hypothesis:
      {src_id, src, hyp_id, tgt, method, M}

    Note: MBR scoring is done per-sentence (CPU), not batched.
    It adds ~1-2 min overhead for 10k sentences.

    CUDA OOM tips
    -------------
    - Reduce pool_size (minimum = M+1)
    - Reduce batch_size
    """
    assert pool_size >= M + 1, "pool_size must be > M to allow re-ranking."

    # Step 1: sample the pool using top-p
    # We collect all pool candidates keyed by src_id first.
    pool = defaultdict(list)     # src_id -> [hypothesis strings]
    src_lookup = {}              # src_id -> src string

    pool_kwargs = dict(
        max_length=max_length,
        forced_bos_token_id=forced_bos_token_id,
        do_sample=True,
        top_p=0.9,               # wider distribution for diverse pool
        num_beams=1,
        num_return_sequences=pool_size,
    )

    for batch_start in tqdm(
        range(0, len(sources), batch_size),
        desc=f"MBR sampling pool={pool_size}",
        unit="batch",
    ):
        batch_srcs = sources[batch_start: batch_start + batch_size]

        inputs = tokenizer(
            batch_srcs,
            return_tensors="pt",
            padding=True,
            truncation=True,
            max_length=max_length,
        ).to(device)

        with torch.no_grad():
            outputs = model.generate(**inputs, **pool_kwargs)

        decoded = tokenizer.batch_decode(outputs, skip_special_tokens=True)

        for i, src in enumerate(batch_srcs):
            src_id = batch_start + i
            src_lookup[src_id] = src
            for k in range(pool_size):
                pool[src_id].append(decoded[i * pool_size + k])

    # Step 2: re-rank each pool and yield top-M
    for src_id in tqdm(sorted(pool.keys()),
                       desc="MBR re-ranking", unit="src"):
        ranked = _mbr_score_pool(pool[src_id])
        for hyp_id, (score, tgt) in enumerate(ranked[:M]):
            yield {
                "src_id": src_id,
                "src":    src_lookup[src_id],
                "hyp_id": hyp_id,
                "tgt":    tgt,
                "method": "mbr",
                "M":      M,
                "mbr_score": round(score, 4),
            }

print("generate_mbr_translations() defined.")

generate_mbr_translations() defined.


## 5. Generate & Save Synthetic Translations

All outputs are written to `data/synthetic/` as JSONL.  
Each line: `{src_id, src, hyp_id, tgt, method, M}`  
MBR rows also include `mbr_score`.

**Resume safety:** each file is written to `.tmp` first and atomically renamed on completion. Interrupted runs leave a `.tmp` behind — delete it and rerun, or set `OVERWRITE = True`.

In [10]:
def save_translations(sources, method, M, out_path,
                      overwrite=False, batch_size=BATCH_SIZE):
    """
    Generate and stream-write hypotheses for non-MBR methods.
    Skips if file exists and overwrite=False.
    """
    out_path = Path(out_path)
    tmp_path = out_path.with_suffix(".tmp")

    if out_path.exists() and not overwrite:
        print(f"  SKIP (exists): {out_path.name}")
        return

    print(f"  Generating {method} M={M} → {out_path.name}")
    total_rows = len(sources) * M

    with open(tmp_path, "w", encoding="utf-8") as fout:
        pbar = tqdm(total=total_rows, desc=f"{method}/M={M}", unit="hyp")
        for row in generate_translations(
            sources, method=method, M=M, batch_size=batch_size
        ):
            fout.write(json.dumps(row, ensure_ascii=False) + "\n")
            pbar.update(1)
            if row["hyp_id"] == M - 1:
                fout.flush()   # flush after last hyp of each source
        pbar.close()

    shutil.move(str(tmp_path), str(out_path))
    print(f"  Saved {total_rows:,} rows → {out_path.name}")


def save_mbr_translations(sources, M, out_path,
                           pool_size=MBR_POOL_SIZE,
                           overwrite=False, batch_size=BATCH_SIZE):
    """
    MBR-specific save: sample pool → re-rank → write top-M per source.
    """
    out_path = Path(out_path)
    tmp_path = out_path.with_suffix(".tmp")

    if out_path.exists() and not overwrite:
        print(f"  SKIP (exists): {out_path.name}")
        return

    print(f"  Generating MBR M={M} pool={pool_size} → {out_path.name}")

    with open(tmp_path, "w", encoding="utf-8") as fout:
        for row in generate_mbr_translations(
            sources, M=M, pool_size=pool_size, batch_size=batch_size
        ):
            fout.write(json.dumps(row, ensure_ascii=False) + "\n")
            fout.flush() if row["hyp_id"] == M - 1 else None

    shutil.move(str(tmp_path), str(out_path))
    total_rows = len(sources) * M
    print(f"  Saved {total_rows:,} rows → {out_path.name}")


# ---------------------------------------------------------------------------
# Run all jobs
# ---------------------------------------------------------------------------
for method, M, fname in JOBS:
    out_path = SYNTHETIC_DIR / fname
    if method == "mbr":
        save_mbr_translations(
            sources, M=M, out_path=out_path,
            pool_size=MBR_POOL_SIZE,
            overwrite=OVERWRITE, batch_size=BATCH_SIZE,
        )
    else:
        save_translations(
            sources, method=method, M=M, out_path=out_path,
            overwrite=OVERWRITE, batch_size=BATCH_SIZE,
        )

print("\nAll generation jobs complete.")

  SKIP (exists): eng_swh_beam_M1.jsonl
  SKIP (exists): eng_swh_beam_M10.jsonl
  SKIP (exists): eng_swh_top_p_M10.jsonl
  SKIP (exists): eng_swh_top_k_M10.jsonl
  SKIP (exists): eng_swh_dbs_M10.jsonl
  SKIP (exists): eng_swh_mbr_M10.jsonl

All generation jobs complete.


## 6. Diversity Analysis

### Self-BLEU — definition

For each source sentence with M hypotheses `{h_0 … h_{M-1}}`:
- For each `h_i`: compute `sentence_bleu(h_i, [h_j | j≠i])`.
- Average over all `i` → source-level self-BLEU.

Corpus self-BLEU = mean over all source sentences.

| Self-BLEU | Meaning |
|-----------|----------|
| Low  | Hypotheses are diverse — good for MHD |
| High | Hypotheses are repetitive — common in standard beam |
| NaN  | M=1, nothing to compare |

> Low self-BLEU ≠ better translation quality. Quality is evaluated separately with FLORES.

In [11]:
def compute_self_bleu(hyp_groups):
    """
    hyp_groups: dict {src_id: [hyp_str, ...]}
    Returns corpus-level self-BLEU (float) or NaN.
    """
    src_bleus = []
    for hyps in hyp_groups.values():
        if len(hyps) < 2:
            continue
        scores = [
            sacrebleu.sentence_bleu(
                hyps[i],
                [hyps[j] for j in range(len(hyps)) if j != i]
            ).score
            for i in range(len(hyps))
        ]
        src_bleus.append(sum(scores) / len(scores))
    return sum(src_bleus) / len(src_bleus) if src_bleus else float("nan")


def analyze_jsonl(jsonl_path):
    """Compute diversity metrics for one synthetic JSONL file."""
    jsonl_path = Path(jsonl_path)
    rows = []
    with open(jsonl_path, encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if line:
                rows.append(json.loads(line))
    if not rows:
        return None

    method    = rows[0]["method"]
    M         = rows[0]["M"]
    src_ids   = [r["src_id"] for r in rows]
    tgt_texts = [r["tgt"]    for r in rows]

    total_rows        = len(rows)
    unique_sources    = len(set(src_ids))
    unique_targets    = len(set(tgt_texts))
    duplicate_rate    = round(1 - unique_targets / total_rows, 4)
    lengths           = [len(t.split()) for t in tgt_texts]
    avg_target_length = round(sum(lengths) / len(lengths), 2)
    all_words         = [w for t in tgt_texts for w in t.split()]
    word_counter      = Counter(all_words)
    unique_words      = len(word_counter)
    top20_words       = word_counter.most_common(20)
    hyp_per_src       = Counter(src_ids)
    avg_hyp_per_src   = round(sum(hyp_per_src.values()) / len(hyp_per_src), 2)

    hyp_groups = defaultdict(list)
    for r in rows:
        hyp_groups[r["src_id"]].append(r["tgt"])

    print(f"  Computing self-BLEU for {jsonl_path.name} ...")
    self_bleu = compute_self_bleu(hyp_groups)

    return {
        "file":                   jsonl_path.name,
        "method":                 method,
        "M":                      M,
        "total_rows":             total_rows,
        "unique_sources":         unique_sources,
        "unique_targets":         unique_targets,
        "duplicate_rate":         duplicate_rate,
        "avg_target_length":      avg_target_length,
        "unique_target_words":    unique_words,
        "avg_hypotheses_per_src": avg_hyp_per_src,
        "self_bleu":              round(self_bleu, 4) if not math.isnan(self_bleu) else None,
        "_top20_words":           top20_words,
    }

print("Analysis helpers defined.")

Analysis helpers defined.


In [12]:
jsonl_files = sorted(SYNTHETIC_DIR.glob("*.jsonl"))
print(f"Found {len(jsonl_files)} JSONL file(s)\n")

analysis_results = []
top_words_store  = {}

for jf in jsonl_files:
    print(f"Analysing: {jf.name}")
    metrics = analyze_jsonl(jf)
    if metrics is None:
        print(f"  WARNING: empty — skipping {jf.name}")
        continue
    top_words_store[jf.name] = metrics.pop("_top20_words")
    analysis_results.append(metrics)
    print(f"  rows={metrics['total_rows']:,}  "
          f"unique_tgt={metrics['unique_targets']:,}  "
          f"dup={metrics['duplicate_rate']:.2%}  "
          f"self_bleu={metrics['self_bleu']}\n")

df_analysis = pd.DataFrame(analysis_results)
if not df_analysis.empty:
    print(df_analysis[["file","method","M","duplicate_rate","self_bleu"]].to_string(index=False))

Found 6 JSONL file(s)

Analysing: eng_swh_beam_M1.jsonl
  Computing self-BLEU for eng_swh_beam_M1.jsonl ...
  rows=10,000  unique_tgt=9,992  dup=0.08%  self_bleu=None

Analysing: eng_swh_beam_M10.jsonl
  Computing self-BLEU for eng_swh_beam_M10.jsonl ...
  rows=100,000  unique_tgt=84,813  dup=15.19%  self_bleu=95.3689

Analysing: eng_swh_dbs_M10.jsonl
  Computing self-BLEU for eng_swh_dbs_M10.jsonl ...
  rows=100,000  unique_tgt=87,858  dup=12.14%  self_bleu=94.8665

Analysing: eng_swh_mbr_M10.jsonl
  Computing self-BLEU for eng_swh_mbr_M10.jsonl ...
  rows=100,000  unique_tgt=99,911  dup=0.09%  self_bleu=70.5242

Analysing: eng_swh_top_k_M10.jsonl
  Computing self-BLEU for eng_swh_top_k_M10.jsonl ...
  rows=100,000  unique_tgt=99,375  dup=0.62%  self_bleu=45.6976

Analysing: eng_swh_top_p_M10.jsonl
  Computing self-BLEU for eng_swh_top_p_M10.jsonl ...
  rows=100,000  unique_tgt=96,022  dup=3.98%  self_bleu=67.8891

                   file method  M  duplicate_rate  self_bleu
  eng_swh

### Save analysis CSV & top-words JSON

In [13]:
csv_cols = [
    "file", "method", "M", "total_rows", "unique_sources",
    "unique_targets", "duplicate_rate", "avg_target_length",
    "unique_target_words", "avg_hypotheses_per_src", "self_bleu",
]

if not df_analysis.empty:
    analysis_csv = RESULTS_DIR / "synthetic_analysis.csv"
    df_analysis[csv_cols].to_csv(analysis_csv, index=False)
    print(f"Saved: {analysis_csv}")

    top_words_json = RESULTS_DIR / "synthetic_top_words.json"
    serialisable = {
        fname: [{"word": w, "count": c} for w, c in pairs]
        for fname, pairs in top_words_store.items()
    }
    with open(top_words_json, "w", encoding="utf-8") as f:
        json.dump(serialisable, f, ensure_ascii=False, indent=2)
    print(f"Saved: {top_words_json}")
else:
    print("No analysis data yet — run generation cells first.")

Saved: C:\Users\nirmi\Desktop\MHD2\results\synthetic_analysis.csv
Saved: C:\Users\nirmi\Desktop\MHD2\results\synthetic_top_words.json


## 7. Plots

Four bar charts comparing all six methods. Saved to `results/plots/`. Matplotlib only.

In [14]:
def bar_plot(df, y_col, ylabel, title, filename, color="steelblue"):
    """Save a bar chart of metric y_col grouped by method/M."""
    if df.empty or y_col not in df.columns:
        print(f"  No data for plot: {filename}")
        return

    labels = [f"{r['method']}\nM={r['M']}" for _, r in df.iterrows()]
    values = df[y_col].tolist()

    fig, ax = plt.subplots(figsize=(max(8, len(labels) * 1.6), 5))
    bars = ax.bar(labels, values, color=color, edgecolor="black", width=0.55)

    for bar, val in zip(bars, values):
        if val is not None and not (isinstance(val, float) and math.isnan(val)):
            ax.text(
                bar.get_x() + bar.get_width() / 2,
                bar.get_height() * 1.01,
                f"{val:.2f}" if isinstance(val, float) else str(val),
                ha="center", va="bottom", fontsize=9,
            )

    ax.set_xlabel("Decoding Method / M", fontsize=11)
    ax.set_ylabel(ylabel, fontsize=11)
    ax.set_title(title, fontsize=12)
    ax.margins(y=0.18)
    plt.tight_layout()
    out = PLOTS_DIR / filename
    fig.savefig(out, dpi=150)
    plt.close(fig)
    print(f"  Saved: {out.name}")


if not df_analysis.empty:
    bar_plot(df_analysis, "self_bleu",
             ylabel="Self-BLEU (lower = more diverse)",
             title="Self-BLEU by Decoding Method / M",
             filename="self_bleu_by_method.png",
             color="steelblue")

    bar_plot(df_analysis, "unique_target_words",
             ylabel="Unique Target Words",
             title="Unique Target Vocabulary by Decoding Method / M",
             filename="unique_words_by_method.png",
             color="darkorange")

    bar_plot(df_analysis, "duplicate_rate",
             ylabel="Duplicate Hypothesis Rate",
             title="Duplicate Rate by Decoding Method / M",
             filename="duplicate_rate_by_method.png",
             color="tomato")

    bar_plot(df_analysis, "avg_target_length",
             ylabel="Avg Target Length (words)",
             title="Average Target Length by Decoding Method / M",
             filename="avg_length_by_method.png",
             color="mediumseagreen")
else:
    print("No analysis data — run generation + analysis cells first.")

  Saved: self_bleu_by_method.png
  Saved: unique_words_by_method.png
  Saved: duplicate_rate_by_method.png
  Saved: avg_length_by_method.png


## 8. (Optional) Teacher Evaluation on FLORES Devtest

Translates `eng_Latn.devtest` with the teacher model using **all six decoding methods** (beam M=1, beam M=10, top-p M=10, top-k M=10, DBS M=10, MBR M=10) and scores each against `swh_Latn.devtest` using **sacreBLEU** and **chrF++**.

For multi-hypothesis methods (M>1) the **first hypothesis** (hyp_id=0) is used as the translation candidate — it is the highest-ranked output from each method.

Results are saved to `results/teacher_flores_scores.csv` and printed as a ranked table.

> Set `RUN_TEACHER_FLORES_EVAL = True` to activate. Adds ~15–25 min on a Kaggle T4 (6 methods × ~1k sentences).  
> Do **not** use FLORES as training data.

In [18]:
RUN_TEACHER_FLORES_EVAL = False   # <-- set True to run

In [19]:
if RUN_TEACHER_FLORES_EVAL:
    import time

    flores_src_path = FLORES_DIR / "eng_Latn.devtest"
    flores_ref_path = FLORES_DIR / "swh_Latn.devtest"
    assert flores_src_path.exists(), f"Missing: {flores_src_path}"
    assert flores_ref_path.exists(), f"Missing: {flores_ref_path}"

    with open(flores_src_path, encoding="utf-8") as f:
        flores_src = [l.strip() for l in f if l.strip()]
    with open(flores_ref_path, encoding="utf-8") as f:
        flores_ref = [l.strip() for l in f if l.strip()]

    assert len(flores_src) == len(flores_ref), "Source/reference length mismatch!"
    print(f"FLORES devtest: {len(flores_src)} source sentences")
    print(f"Evaluating {len(JOBS)} method(s)...\n")

    flores_records = []

    for method, M, fname in JOBS:
        label = f"{method}/M={M}"
        t0 = time.time()
        print(f"  [{label}] translating ...", end=" ", flush=True)

        # Collect only hyp_id=0 (best/first hypothesis) per source sentence
        hyp0 = {}   # src_id -> tgt string

        if method == "mbr":
            # MBR: sample pool then re-rank — collect hyp_id=0 per source
            for row in generate_mbr_translations(
                flores_src,
                M=M,
                pool_size=MBR_POOL_SIZE,
                batch_size=BATCH_SIZE,
                max_length=MAX_LENGTH,
            ):
                if row["hyp_id"] == 0:
                    hyp0[row["src_id"]] = row["tgt"]
        else:
            for row in generate_translations(
                flores_src,
                method=method,
                M=M,
                batch_size=BATCH_SIZE,
                max_length=MAX_LENGTH,
            ):
                if row["hyp_id"] == 0:
                    hyp0[row["src_id"]] = row["tgt"]

        # Reconstruct hypothesis list in order
        flores_hyps = [hyp0[i] for i in range(len(flores_src))]

        elapsed = time.time() - t0

        # sacreBLEU
        bleu_result = sacrebleu.corpus_bleu(flores_hyps, [flores_ref])
        # chrF++ (beta=2, char_order=6, word_order=2)
        chrf_result = sacrebleu.corpus_chrf(
            flores_hyps, [flores_ref],
            char_order=6, word_order=2, beta=2,
        )

        print(f"BLEU={bleu_result.score:.2f}  chrF++={chrf_result.score:.2f}  "
              f"({elapsed/60:.1f} min)")

        flores_records.append({
            "model":    TEACHER_MODEL,
            "src_lang": SRC_LANG,
            "tgt_lang": TGT_LANG,
            "eval_set": "flores_devtest",
            "method":   method,
            "M":        M,
            "sacrebleu":  round(bleu_result.score, 4),
            "chrf_pp":    round(chrf_result.score, 4),
        })

    # ── Save CSV ──────────────────────────────────────────────────────────────
    df_flores = pd.DataFrame(flores_records)
    flores_csv = RESULTS_DIR / "teacher_flores_scores.csv"
    df_flores.to_csv(flores_csv, index=False)
    print(f"\nSaved: {flores_csv}")

    # ── Pretty table ──────────────────────────────────────────────────────────
    print("\n" + "=" * 52)
    print(f"{'Method':<12} {'M':>4}  {'sacreBLEU':>10}  {'chrF++':>8}")
    print("-" * 52)
    for rec in sorted(flores_records, key=lambda r: r["sacrebleu"], reverse=True):
        print(f"{rec['method']:<12} {rec['M']:>4}  "
              f"{rec['sacrebleu']:>10.2f}  {rec['chrf_pp']:>8.2f}")
    print("=" * 52)
    print("(Ranked by sacreBLEU descending)")

else:
    print("FLORES eval skipped (RUN_TEACHER_FLORES_EVAL = False).")


FLORES devtest: 1012 sentences


FLORES beam M=1:   0%|          | 0/1012 [00:00<?, ?it/s]

FLORES beam M=1: 100%|██████████| 1012/1012 [05:06<00:00,  3.30it/s]



FLORES devtest — facebook/nllb-200-distilled-600M
  sacreBLEU : 32.49
  chrF      : 60.97
  Saved: teacher_flores_scores.csv


## 9. Final Summary

In [20]:
print("=" * 65)
print("PHASE 2 — FINAL SUMMARY")
print("=" * 65)

print("\n[1] Synthetic files created:")
for jf in sorted(SYNTHETIC_DIR.glob("*.jsonl")):
    size_mb = jf.stat().st_size / 1_048_576
    print(f"     {jf.name:<40}  {size_mb:.1f} MB")

print("\n[2] Self-BLEU per method (lower = more diverse):")
if not df_analysis.empty:
    best_bleu = float("inf")
    best_label = ""
    for _, row in df_analysis.iterrows():
        sb = row["self_bleu"]
        sb_str = f"{sb:.4f}" if sb is not None else "N/A (M=1)"
        label  = f"{row['method']:<8} M={row['M']}"
        print(f"     {label:<15}  self-BLEU = {sb_str}")
        if sb is not None and sb < best_bleu:
            best_bleu  = sb
            best_label = label
    if best_label:
        print(f"\n[3] Most diverse method: {best_label.strip()}  (self-BLEU = {best_bleu:.4f})")
else:
    print("     No data — run generation and analysis cells.")

print("""
[!] REMINDER
    Low self-BLEU = diverse hypotheses.
    This does NOT guarantee better translation quality.
    Quality is measured separately via FLORES BLEU / chrF.

    MBR outputs are the *consensus best* hypotheses from a
    sampled pool — highest quality but least diverse.
    DBS outputs are *structurally diverse* — good for coverage.

[→] NEXT — Phase 3:
    Use the JSONL files in data/synthetic/ to train student
    models via multi-hypothesis distillation (MHD).
    Do NOT use FLORES as training data.
""")
print("=" * 65)

PHASE 2 — FINAL SUMMARY

[1] Synthetic files created:
     eng_swh_beam_M1.jsonl                     3.4 MB
     eng_swh_beam_M10.jsonl                    34.1 MB
     eng_swh_dbs_M10.jsonl                     34.3 MB
     eng_swh_mbr_M10.jsonl                     35.6 MB
     eng_swh_top_k_M10.jsonl                   34.3 MB
     eng_swh_top_p_M10.jsonl                   33.9 MB

[2] Self-BLEU per method (lower = more diverse):
     beam     M=1     self-BLEU = nan
     beam     M=10    self-BLEU = 95.3689
     dbs      M=10    self-BLEU = 94.8665
     mbr      M=10    self-BLEU = 70.5242
     top_k    M=10    self-BLEU = 45.6976
     top_p    M=10    self-BLEU = 67.8891

[3] Most diverse method: top_k    M=10  (self-BLEU = 45.6976)

[!] REMINDER
    Low self-BLEU = diverse hypotheses.
    This does NOT guarantee better translation quality.
    Quality is measured separately via FLORES BLEU / chrF.

    MBR outputs are the *consensus best* hypotheses from a
    sampled pool — highest 